# Web Scraping with BeautifulSoup

**BeautifulSoup** parses HTML and XML documents and lets you navigate and search the tree.

Install: `pip install beautifulsoup4 lxml`

### Ethical Scraping Rules
- Check the site's `robots.txt` before scraping
- Don't overwhelm servers (add delays)
- Respect terms of service
- Cache results to avoid re-requesting

In this notebook we use **inline HTML strings** (no real HTTP requests needed).

In [ ]:
from bs4 import BeautifulSoup
print('BeautifulSoup ready!')

## 1. Parsing HTML

In [ ]:
from bs4 import BeautifulSoup

html = '''
<html>
<head><title>My Blog</title></head>
<body>
  <h1 id="main-title">Welcome to My Blog</h1>
  <p class="intro">This is my <strong>awesome</strong> blog.</p>
  <p class="intro">It covers Python and web development.</p>
  <a href="/about">About Me</a>
  <a href="https://python.org" class="external">Python.org</a>
</body>
</html>
'''

soup = BeautifulSoup(html, 'html.parser')

# Get title
print(soup.title.string)         # 'My Blog'
print(soup.h1.text)              # 'Welcome to My Blog'
print(soup.find('p').text)       # first <p>
print(soup.find('a')['href'])    # '/about' — get attribute

## 2. `find()` and `find_all()`

In [ ]:
html = '''
<div class="container">
  <h2>Top Python Libraries</h2>
  <ul id="library-list">
    <li class="item popular"><a href="/numpy">NumPy</a> - numerical computing</li>
    <li class="item"><a href="/pandas">Pandas</a> - data analysis</li>
    <li class="item popular"><a href="/requests">Requests</a> - HTTP</li>
    <li class="item"><a href="/flask">Flask</a> - web framework</li>
  </ul>
</div>
'''

soup = BeautifulSoup(html, 'html.parser')

# find() — first match
first_li = soup.find('li')
print('First li:', first_li.get_text())

# find_all() — all matches
all_li = soup.find_all('li')
print(f'\nAll items ({len(all_li)}):')
for li in all_li:
    print(f'  {li.get_text().strip()}')

# Find by class
popular = soup.find_all('li', class_='popular')
print(f'\nPopular ({len(popular)}):')
for item in popular:
    print(f'  {item.a.text}')

## 3. CSS Selectors with `select()`

In [ ]:
html = '''
<article class="post">
  <h1 class="title">Python Tips</h1>
  <div class="meta">
    <span class="author">Alice</span>
    <span class="date">2024-01-15</span>
  </div>
  <div class="tags">
    <a href="/tag/python" class="tag">Python</a>
    <a href="/tag/tips" class="tag">Tips</a>
  </div>
  <p class="content">Here are some Python tips.</p>
</article>
'''

soup = BeautifulSoup(html, 'html.parser')

# CSS selectors
print(soup.select_one('h1.title').text)          # element with class
print(soup.select_one('.meta .author').text)     # nested selector
print(soup.select_one('span.date').text)         # by tag + class

# select() returns a list
tags = soup.select('a.tag')
print(f'Tags: {[t.text for t in tags]}')

# Attribute selectors
hrefs = soup.select('a[href]')
print(f'Links: {[a["href"] for a in hrefs]}')

## 4. Navigating the Tree

In [ ]:
html = '''
<div id="parent">
  <p id="first">First paragraph</p>
  <p id="second">Second paragraph</p>
  <p id="third">Third paragraph</p>
</div>
'''

soup = BeautifulSoup(html, 'html.parser')
div = soup.find('div')

# Children
children = list(div.children)  # includes whitespace text nodes
paragraphs = list(div.find_all('p'))
print(f'Paragraphs: {len(paragraphs)}')

# Navigation from a specific element
second = soup.find('p', id='second')
print(f'Parent tag: {second.parent.name}')
print(f'Previous sibling text: {second.find_previous_sibling("p").get_text()}')
print(f'Next sibling text: {second.find_next_sibling("p").get_text()}')

## 5. Extracting Text and Attributes

In [ ]:
html = '''
<table id="products">
  <thead>
    <tr><th>Product</th><th>Price</th><th>Stock</th></tr>
  </thead>
  <tbody>
    <tr><td>Laptop</td><td>£999</td><td data-qty="5">In Stock</td></tr>
    <tr><td>Phone</td><td>£699</td><td data-qty="12">In Stock</td></tr>
    <tr><td>Tablet</td><td>£449</td><td data-qty="0">Out of Stock</td></tr>
  </tbody>
</table>
'''

soup = BeautifulSoup(html, 'html.parser')

# Parse a table
rows = soup.select('#products tbody tr')
print(f'{"Product":10} {"Price":8} {"Qty":5} {"Status"}')
print('-' * 35)
for row in rows:
    cells = row.find_all('td')
    product = cells[0].text
    price = cells[1].text
    qty = cells[2].get('data-qty', '?')  # get custom attribute
    status = cells[2].text
    print(f'{product:10} {price:8} {qty:5} {status}')

## 6. Combining with `requests`

In [ ]:
# Real-world pattern: requests + BeautifulSoup
# (Using a mock HTML response here to avoid live HTTP)

import requests
from bs4 import BeautifulSoup

def scrape_page(url):
    """Fetch a URL and return a BeautifulSoup object."""
    headers = {'User-Agent': 'Mozilla/5.0 (educational scraper)'}
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    return BeautifulSoup(response.text, 'html.parser')

# Simulate the pattern without making real requests
def simulate_scrape():
    # Simulated HTML (as if fetched from a website)
    html = '''
    <html>
    <head><title>Books - Fake Store</title></head>
    <body>
      <div class="book" data-id="1">
        <h3>Clean Code</h3>
        <span class="price">£35.99</span>
        <span class="author">Robert C. Martin</span>
      </div>
      <div class="book" data-id="2">
        <h3>The Pragmatic Programmer</h3>
        <span class="price">£42.00</span>
        <span class="author">David Thomas</span>
      </div>
      <div class="book" data-id="3">
        <h3>Python Cookbook</h3>
        <span class="price">£38.50</span>
        <span class="author">David Beazley</span>
      </div>
    </body>
    </html>
    '''
    
    soup = BeautifulSoup(html, 'html.parser')
    books = []
    
    for book_div in soup.select('.book'):
        book_id = book_div.get('data-id')
        title = book_div.find('h3').text
        price = book_div.find(class_='price').text
        author = book_div.find(class_='author').text
        books.append({'id': book_id, 'title': title, 'price': price, 'author': author})
    
    return books

books = simulate_scrape()
print(f'{'ID':3} {"Title":30} {"Price":8} Author')
print('-' * 65)
for b in books:
    print(f"{b['id']:3} {b['title']:30} {b['price']:8} {b['author']}")

## Mini-Project: News Aggregator (Simulated)

In [ ]:
from bs4 import BeautifulSoup
from datetime import datetime

# Simulated HTML from a news site
news_html = '''
<html>
<body>
  <section class="news-feed">
    <article class="news-item" data-category="tech">
      <h2><a href="/tech/python-4">Python 4.0 Announced with Major Performance Improvements</a></h2>
      <p class="summary">The Python Software Foundation announced Python 4.0 will focus on speed improvements and better type system integration.</p>
      <span class="author">Jane Doe</span>
      <time datetime="2024-01-15">January 15, 2024</time>
      <span class="reading-time">3 min read</span>
    </article>
    <article class="news-item" data-category="ai">
      <h2><a href="/ai/new-model">New AI Model Breaks Multiple Benchmarks</a></h2>
      <p class="summary">A research team unveiled a new language model that surpasses previous state-of-the-art on key benchmarks.</p>
      <span class="author">John Smith</span>
      <time datetime="2024-01-14">January 14, 2024</time>
      <span class="reading-time">5 min read</span>
    </article>
    <article class="news-item" data-category="web">
      <h2><a href="/web/framework-update">Popular Web Framework Releases v3.0</a></h2>
      <p class="summary">The latest major release brings async support, improved routing, and built-in rate limiting.</p>
      <span class="author">Alice Brown</span>
      <time datetime="2024-01-13">January 13, 2024</time>
      <span class="reading-time">4 min read</span>
    </article>
  </section>
</body>
</html>
'''

def parse_news(html):
    """Parse news articles from HTML."""
    soup = BeautifulSoup(html, 'html.parser')
    articles = []
    
    for article in soup.select('.news-item'):
        link = article.find('a')
        time_el = article.find('time')
        
        articles.append({
            'title': link.text if link else 'Untitled',
            'url': link.get('href', '') if link else '',
            'summary': article.find(class_='summary').text if article.find(class_='summary') else '',
            'author': article.find(class_='author').text if article.find(class_='author') else 'Unknown',
            'date': time_el.get('datetime', '') if time_el else '',
            'reading_time': article.find(class_='reading-time').text if article.find(class_='reading-time') else '',
            'category': article.get('data-category', ''),
        })
    
    return articles


articles = parse_news(news_html)

print('=== NEWS DIGEST ===')
print()
for a in articles:
    print(f"[{a['category'].upper()}] {a['title']}")
    print(f"  {a['summary'][:80]}...")
    print(f"  By {a['author']} | {a['date']} | {a['reading_time']}")
    print()